# **PicassoPy Workshop --> Test case: cpv 2024-08-21**
---


- Folder for data `test_case_data` -> download separately
- Folder for configs `test_case_config`


## Imports

In [1]:
import os, sys
import argparse
import datetime
import logging
from pathlib import Path
import numpy as np

sys.path.append('../')
import ppcpy
import ppcpy.io.loadConfigs as loadConfigs
import ppcpy.io.readPollyRawData as readPollyRawData
import ppcpy.interface.picassoProc as picassoProc
import ppcpy.misc.helper as helper
import ppcpy.misc.startscreen as startscreen
from ppcpy.io.write2nc import write_channelwise_2_nc_file, write2nc_file, write_profile2nc_file

import matplotlib
import matplotlib.pyplot as plt
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=plt.cm.tab20.colors)

## Defining Script Inputs

- The parameters `args.device`, `args.timestamp`, `args.picasso_config_file`, `args.level0_file_to_process`, need to be manually specified per case.
- The parameter `DATABASE_PATH` is the path to the database used for storing the retrieved calibration constants

In [2]:
## For purpose of the notebook mimic the argparse interface
from types import SimpleNamespace
args = SimpleNamespace()

## The used device and time of measurement
args.device = 'pollyxt_cpv'
args.timestamp = '20240821'
dt = datetime.datetime.strptime(args.timestamp, "%Y%m%d")

## The used config file
args.picasso_config_file = "test_case_config/pollynet_processing_chain_config_test.json"

## The data file to use
args.level0_file_to_process = f"test_case_data/{dt:%Y_%m_%d_%a}_CPV_00_00_01.nc"


## Database path
DATABASE_PATH = "PicassoPyDatabase.db"

In [3]:
startscreen.startscreen()

      ____  _                            ____           ___ ____ 
     / __ \(_)________ _______________  / __ \__  __   <  // __ \
    / /_/ / / ___/ __ `/ ___/ ___/ __ \/ /_/ / / / /   / // / / /
   / ____/ / /__/ /_/ (__  |__  ) /_/ / ____/ /_/ /   / // /_/ / 
  /_/   /_/\___/\__,_/____/____/\____/_/    \__, /   /_(_)____/  
                                           /____/                


## Load Data and Config-files

Loads data and information from the config files into the following three dictionaries:
- `picasso_config_dict`: Paths and other information stored in the picasso config file
- `polly_config_dict`: Configuration variables from polly config and polly default files
- `rawdata_dict`: Measurement data and information extracted from the level0 file

In [4]:
## Path to default Picasso config file
picasso_default_config_file = Path(
    helper.detect_path_type(Path.cwd().parent), 'ppcpy', 'config', 'pollynet_processing_chain_config.json')

## Load Picasso config file
picasso_config_dict = loadConfigs.loadPicassoConfig(args.picasso_config_file, picasso_default_config_file)

## load polly config file
polly_config_array = loadConfigs.readPollyNetConfigLinkTable(
    picasso_config_dict['pollynet_config_link_file'], timestamp=args.timestamp, device=args.device
)
polly_config_dict = loadConfigs.getPollyConfigfromArray(
    polly_config_array, picasso_config_dict
)

## Load level0-data file
rawfile_fullname = args.level0_file_to_process
rawfile = helper.detect_path_type(rawfile_fullname)
rawdata_dict = readPollyRawData.readPollyRawData(rawfile)

2026-08-13 17:13:22,457 - INFO - picasso_default_config_file: c:\Users\buholdt\Documents\PicassoPy\ppcpy\config\pollynet_processing_chain_config.json
2026-08-13 17:13:22,460 - INFO - picasso_config_file: test_case_config/pollynet_processing_chain_config_test.json
2026-08-13 17:13:22,462 - INFO - pollynet_config_link_file: test_case_config/pollynet_processing_chain_config_links.xlsx
2026-08-13 17:13:23,018 - INFO - polly_default_config_file: c:\Users\buholdt\Documents\PicassoPy\ppcpy\config\polly_global_config.json
2026-08-13 17:13:23,020 - INFO - polly_config_file: test_case_config\pollyxt_cpv_config_20230927.json
2026-08-13 17:13:23,023 - INFO - keys default/template file, but not in specific file {'zLim_NR_RCS_407', 'minSNR_4_sigNorm', 'turbid_thres_par_beta_1064', 'droplet_thres_par_depol', 'yLim_beta_532_Poliphon', 'volDepolerror532', 'ice_thres_par_depol', 'imgFormat', 'polCaliEtaStd532', 'xLim_Profi_LR', 'flagUseImprovedSNR', 'isParallel', 'bgCorRangeIndxLow', 'flagSigTempCor', '

## Initialize PicassoProc object

PicassoProc is the main object in PicassoPy, and is responsible for running all processes included and storing the data.

In [5]:
## Initialize PicassoProc
data_cube = picassoProc.PicassoProc(rawdata_dict, polly_config_dict, picasso_config_dict)

In [6]:
## reset date if date in filename differs date within nc-file 
data_cube.reset_date_infile()

## checking for correct measurement shots
data_cube.check_for_correct_mshots()

## setting channelTags
data_cube.setChannelTags()

## check for correct date in nc-file
data_cube.reset_date_infile()

2026-08-13 17:13:25,390 - INFO - date consistency-check... 
2026-08-13 17:13:25,391 - INFO - ... date in nc-file equals date of filename
2026-08-13 17:13:25,393 - INFO - ChannelLabels: ['FR-total-355 nm', 'FR-cross-355 nm', 'FR-387 nm', 'FR-407 nm', 'FR-total-532 nm', 'FR-cross-532 nm', 'FR-607 nm', 'FR-total-1064 nm', 'NR-total-532 nm', 'NR-607 nm', 'NR-total-355 nm', 'NR-387 nm', 'DFOV', '1058', '1064s', 'none']
2026-08-13 17:13:25,394 - WARNING - removed none tag from channel list [15]
2026-08-13 17:13:25,395 - INFO - date consistency-check... 
2026-08-13 17:13:25,396 - INFO - ... date in nc-file equals date of filename


## Preprocessing & Saturation Detection

The preprocessing includes the following processes:
- Deadtime correction
- Background correction
- SNR claculations
- Flagging of data
- Range correction

In [7]:
## Perform preprocessing, this includes Dead-time correction, Background correction, and Range correction
data_cube.preprocessing(collect_debug=True)

2026-08-13 17:13:25,406 - INFO - Preprocessing ...
2026-08-13 17:13:25,407 - INFO - ... time conversion
2026-08-13 17:13:25,415 - WARNING - ... mShots not constant min 0 max 2999
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\preprocess\pollyPreprocess.py:507: RuntimeWarning: invalid value encountered in divide
  PCR = (c * signal)/(2 * hRes * mShots[:, np.newaxis, :])
2026-08-13 17:13:26,495 - INFO - ... calculate dead-time corrected signal
2026-08-13 17:13:26,496 - INFO - ... Deadtime-correction (Mode: 1)
2026-08-13 17:13:38,132 - INFO - ... calculate background corrected signal
2026-08-13 17:13:38,133 - INFO - ... removing background from signal
2026-08-13 17:13:40,456 - INFO - ... height bin calculations
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\preprocess\pollyPreprocess.py:378: UserWarning: no explicit representation of timezones available for np.datetime64
  data_dict['time64'] = np.array([np.datetime64(t) for t in mTime_obj])
2026-08-13 17:13:40,726 - INFO - ... 

In [8]:
## Save high resolution signal-to-noise ratio, background, and range corrected signal
write_channelwise_2_nc_file(data_cube=data_cube, prod_ls=['SNR', 'BG', 'RCS'])

2026-08-13 17:13:47,100 - INFO - saving product: SNR
2026-08-13 17:13:47,232 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_SNR.nc
2026-08-13 17:14:05,376 - INFO - saving product: BG
2026-08-13 17:14:05,486 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_BG.nc
2026-08-13 17:14:05,511 - INFO - saving product: RCS
2026-08-13 17:14:05,610 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_RCS.nc


In [9]:
## Display available channels
data_cube.channel_dict

{0: 'FR-total-355 nm',
 1: 'FR-cross-355 nm',
 2: 'FR-387 nm',
 3: 'FR-407 nm',
 4: 'FR-total-532 nm',
 5: 'FR-cross-532 nm',
 6: 'FR-607 nm',
 7: 'FR-total-1064 nm',
 8: 'NR-total-532 nm',
 9: 'NR-607 nm',
 10: 'NR-total-355 nm',
 11: 'NR-387 nm',
 12: 'DFOV',
 13: '1058',
 14: '1064s'}

In [10]:
## Detect and flag saturated signal
data_cube.SaturationDetect()

2026-08-13 17:14:27,057 - INFO - Saturation detection ...
2026-08-13 17:14:27,059 - INFO - Saturation detection


## Depol Calibration


- Depol. calibration constants (DC) are retrieved at each depol. calibration period included in the data
- All retrieved DCs are stored in a dedicated database
- The optimal retrieved DC, ie. the one with the lowest standard deviation (std) is used for the processing
- If no DCs can be retirieved, the DC with the lowest std in the time range [24h before the measurement, 24h after the measurement] included in the database will be used

In [11]:
## Delta 90 polarization calibration
data_cube.polarizationCaliD90()

2026-08-13 17:14:38,230 - INFO - Delta 90 polarization calibration ...
2026-08-13 17:14:38,231 - INFO - Starting loadGHK
2026-08-13 17:14:38,232 - INFO - Using GHK from config file
2026-08-13 17:14:38,234 - INFO - G: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.], H: [ 0.02041 -0.998    1.       1.      -0.01477 -0.9987   1.      -0.02439
  1.       1.       1.       1.       1.       1.      -0.996  ], K: [0.97859 1.      1.      1.      0.99343 1.      1.      0.9947  1.
 1.      1.      1.      1.      1.      1.     ]
2026-08-13 17:14:38,236 - INFO - and even a 355 channel
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\calibration\polarization.py:351: RuntimeWarning: divide by zero encountered in divide
  dplus = smooth_signal(sig_x_p, smooth_win) / smooth_signal(sig_t_p, smooth_win)
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\calibration\polarization.py:351: RuntimeWarning: invalid value encountered in divide
  dplus = smooth_signal(sig_x_p, smooth_win) / smooth_signa

In [12]:
## Display depolarization calibration constants
data_cube.etaused

{'355_FR': np.float64(47.27529323156868),
 '532_FR': np.float64(12.066747512729577),
 '1064_FR': np.float64(0.14061415579754508)}

## Cloud Screening

Three modes of cloud Screening are currently implemented:

0. No cloud screening. Return cloud free for all timestamps
1. Cloud screen with Maximum Gradiant Signal (MSG) algorithm
2. Cloud screen with Zhao's algorithm

Clouds are screened per timestamp (30s). After the screening the data is splitt up into cloud free segments and aggregated.

In [13]:
## Apply cloud screening
data_cube.cloudScreen()

2026-08-13 17:14:38,749 - INFO - Cloud screening ...
2026-08-13 17:14:38,750 - INFO - cloud screen mode 1: MSG method.
2026-08-13 17:14:38,985 - INFO - Skipping cloud screening for timestamp 1502.
2026-08-13 17:14:39,334 - INFO - Skipping cloud screening for timestamp 1502.


In [14]:
## Segmentate cloud free groups
data_cube.cloudFreeSeg()

2026-08-13 17:14:39,469 - INFO - Segment cloud free groups ...
2026-08-13 17:14:39,471 - INFO - intNProfiles: 120, minIntNProfiles: 30


In [15]:
## Display cloud free groups
data_cube.clFreeGrps

array([[  0, 113],
       [144, 244]])

In [16]:
## Aggregate background, background corrected signal, and range corrected signal
data_cube.aggregate_profiles()

2026-08-13 17:14:39,499 - INFO - Aggregating variable: sigBGCor ...
2026-08-13 17:14:39,542 - INFO - Aggregating variable: BG ...
2026-08-13 17:14:39,543 - INFO - Aggregating variable: RCS ...
2026-08-13 17:14:39,581 - INFO - Aggregating variable: mShots ...
2026-08-13 17:14:39,582 - INFO - Aggregating variable: mask387Off ...
2026-08-13 17:14:39,583 - INFO - Aggregating variable: mask607Off ...
2026-08-13 17:14:39,584 - INFO - Aggregating variable: mask407Off ...


## Molecular Profiles

The molecular profiles are calculated from cloudNet ECMWF model data. Gdas1 data is not supported in PicassoPy!

In [17]:
## QuickFix for loadMeteo bug:
METEO_DATA_DIR_PATH = "test_case_data"
data_cube.polly_config_dict['meteorDataSource'] = 'nc_cloudnet'
data_cube.polly_config_dict['meteo_folder'] = METEO_DATA_DIR_PATH
data_cube.polly_config_dict['meteo_file'] = r"[\\/]{0:%Y%m%d}_.*\.nc"

## Load meteorological data
data_cube.loadMeteo()

## Calculate molecular profiles
data_cube.calcMolecular()

2026-08-13 17:14:39,597 - INFO - Loading meteorological data ...
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\io\readMeteo.py:263: FutureWarning: In a future version, xarray will not decode the variable 'forecast_time' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds = xr.load_dataset(filename)
2026-08-13 17:14:39,692 - INFO - Performing model height correction.
2026-08-13 17:14:39,693 - INFO - Uncorrected model height[:,0]:
[10.466079  10.4652    10.460253  10.455192  10.449221  10.447984
 10.453651  10.460172  10.464973  10.472917  10.479133  10.48550

## Rayleigh-Fit

- Douglas-Peucker algorithm is used to segment the signal into potential reference heights
- The reference height with the best fit to the molecular backscatter per channel is chosen
- Currently only done for FR-channels. NR reference heights are read from the config variables `refH_NR_{wavelength}`

In [18]:
## Rayleigh-fit procedure --> produces the reference heights
data_cube.rayleighFit()

2026-08-13 17:14:39,776 - INFO - Start Rayleigh Fit
2026-08-13 17:14:39,777 - WARNING - Potential for differences to matlab code due to numerical issues (subtraction of two small values)
2026-08-13 17:14:39,778 - WARNING - rayleighfit seems to use range in matlab, but the met data should be in height >> RECHECK!
2026-08-13 17:14:39,779 - WARNING - at 10km height this is a difference of about 4 indices
2026-08-13 17:14:39,780 - INFO - Cloud free segment 0, Time: 2024-08-21T00:00:00.000000 - 2024-08-21T00:56:30.000000.
2026-08-13 17:14:39,780 - INFO - Channel: 532 total FR.
2026-08-13 17:14:39,784 - WARNING - Warning: Odd smoothinglengs not allowd, will preform smoothing with length win - 1.
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\calibration\rayleighfit.py:574: RuntimeWarning: divide by zero encountered in divide
  std_aer_norm = sig_aer_norm / np.sqrt(pc + bg)
2026-08-13 17:14:39,841 - INFO - Channel: 355 total FR.
2026-08-13 17:14:39,843 - WARNING - Warning: Odd smoothingl

In [19]:
## Display reference heights for a given cloud free group
grpIdx = 0
print(f"""Reference heights in meters for cloud free period {grpIdx} {data_cube.retrievals_highres['time64'][data_cube.clFreeGrps[grpIdx]]}:
355 total FR:  {np.round(data_cube.retrievals_profile['refH'][grpIdx]["355_total_FR"]['refHeight'], 0)}
532 total FR:  {np.round(data_cube.retrievals_profile['refH'][grpIdx]["532_total_FR"]['refHeight'], 0)}
1064 total FR: {np.round(data_cube.retrievals_profile['refH'][grpIdx]["1064_total_FR"]['refHeight'], 0)}
355 total NR:  {np.round(data_cube.retrievals_profile['refH'][grpIdx]["355_total_NR"]['refHeight'], 0)}
532 total NR:  {np.round(data_cube.retrievals_profile['refH'][grpIdx]["532_total_NR"]['refHeight'], 0)}""")

Reference heights in meters for cloud free period 0 ['2024-08-21T00:00:00.000000' '2024-08-21T00:56:30.000000']:
355 total FR:  [ 6877. 12849.]
532 total FR:  [15768. 17935.]
1064 total FR: [13102. 13512.]
355 total NR:  [3005. 3995.]
532 total NR:  [3005. 3995.]


## GHK-Transmission Correction

In [20]:
## Molecular polarization calibration 
data_cube.polarizationCaliMol()

2026-08-13 17:14:40,124 - WARNING - 'flagMolDepolCali' set to False


In [21]:
## Apply GHK-transmission correction
data_cube.transCor()

2026-08-13 17:14:40,135 - INFO - GHK Transmission correction ...
2026-08-13 17:14:40,320 - INFO - Channel: 355 total FR | 355 cross FR
2026-08-13 17:14:40,472 - INFO - G [1.] [1.] H [0.02041] [-0.998] Eta 47.27529323156868 error [ 0.00016  0.01567 -0.00958] Window 1 
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\depolarization.py:195: RuntimeWarning: divide by zero encountered in divide
  sig_ratio = sigc / sigt
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\depolarization.py:195: RuntimeWarning: invalid value encountered in divide
  sig_ratio = sigc / sigt
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\depolarization.py:205: RuntimeWarning: invalid value encountered in divide
  vol_depol = (sig_ratio / eta * (Gt + Ht) - (Gr + Hr)) / ((Gr - Hr) - sig_ratio / eta * (Gt - Ht))
2026-08-13 17:14:40,885 - INFO - Channel: 532 total FR | 532 cross FR
2026-08-13 17:14:41,039 - INFO - G [1.] [1.] H [-0.01477] [-0.9987] Eta 12.066747512729577 erro

In [ ]:
## Aggregate GHK-transmission corrected profiles
data_cube.aggregate_profiles(var=['sigTCor', 'BGTCor'])

2026-08-13 17:18:47,606 - INFO - Aggregating variable: sigTCor ...
2026-08-13 17:18:47,674 - INFO - Aggregating variable: BGTCor ...


## Klett and Raman retrieval

Produces the following profiles per channel:
- Klett: Aerosol Backscatter and Extinction
- Raman: Aerosol Backscatter, Aerosol Extinction, and Lidar Ratio

If `nr=True`, perform the retrievals for FR and NR channels. Otherwise only FR.

In [23]:
## Klett retrieval for GHK-transmission corrected profiles
data_cube.retrievalKlett(nr=True)

2026-08-13 17:14:42,219 - INFO - Klett retrieval for FR & NR GHK-transmission corrected signal ...
2026-08-13 17:14:42,220 - WARNING - rayleighfit seems to use range in matlab, but the met data should be in height >> RECHECK!
2026-08-13 17:14:42,221 - WARNING - at 10km height this is a difference of about 4 indices
2026-08-13 17:14:42,221 - INFO - Cloud free segment 0, Time: 2024-08-21T00:00:00.000000 - 2024-08-21T00:56:30.000000.
2026-08-13 17:14:42,222 - INFO - Channel: 532, total, FR klett.
2026-08-13 17:14:42,252 - INFO - Channel: 355, total, FR klett.
2026-08-13 17:14:42,279 - INFO - Channel: 1064, total, FR klett.
2026-08-13 17:14:42,308 - INFO - Channel: 532, total, NR klett.
2026-08-13 17:14:42,336 - INFO - Channel: 355, total, NR klett.
2026-08-13 17:14:42,362 - INFO - Cloud free segment 1, Time: 2024-08-21T01:12:00.000000 - 2024-08-21T02:02:00.000000.
2026-08-13 17:14:42,363 - INFO - Channel: 532, total, FR klett.
2026-08-13 17:14:42,391 - INFO - Channel: 355, total, FR klett

In [24]:
## Raman retrieval for GHK-transmission corrected profiles
data_cube.retrievalRaman(nr=True)

2026-08-13 17:14:42,527 - INFO - Raman retrieval for FR & NR GHK-transmission corrected signal ...
2026-08-13 17:14:42,528 - WARNING - rayleighfit seems to use range in matlab, but the met data should be in height >> RECHECK!
2026-08-13 17:14:42,529 - WARNING - at 10km height this is a difference of about 4 indices
2026-08-13 17:14:42,530 - INFO - Cloud free segment 0, Time: 2024-08-21T00:00:00.000000 - 2024-08-21T00:56:30.000000.
2026-08-13 17:14:42,531 - INFO - Channels: 355, total, FR | 387, total, FR raman.
c:\Users\buholdt\AppData\Local\miniconda3\envs\PicassoPy\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
2026-08-13 17:14:42,687 - INFO - Filling aerExt below overlap with 0.00010661377416860892 for calculating the backscatter
2026-08-13 17:14:43,175 - INFO - Channels: 532, total, FR | 607, total, FR raman.
2026-08-13 17:14:43,308 - INFO - Filling aerExt below

## Overlap Correction

Two methods are available for calculating the Overlap Function:

1. FRNR method
2. Raman method

And four methods (currently only 3 implemented) are available for applying the Overlap Correction:

0. no overlap correction
1. overlap correction with using the default overlap function (read function from file)
2. overlap correction with using the calculated overlap function
3. overlap correction with gluing near-range and far-range signal -> Not implemented yet!


In [25]:
## Calculate overlap function
data_cube.overlapCalc()

## Fix spike in lower bins
data_cube.overlapFixLowestBins()

## Apply overlap correction
data_cube.overlapCor()

2026-08-13 17:14:48,434 - INFO - calculating overlap functions ...
2026-08-13 17:14:48,435 - WARNING - rayleighfit seems to use range in matlab, but the met data should be in height >> RECHECK!
2026-08-13 17:14:48,436 - WARNING - at 10km height this is a difference of about 4 indices
2026-08-13 17:14:48,437 - INFO - Starting Overlap retrieval
2026-08-13 17:14:48,438 - INFO - Cloud free segment: 0. Time: 2024-08-21T00:00:00.000000 - 2024-08-21T00:56:30.000000.
2026-08-13 17:14:48,439 - INFO - Channels: 355 total FR | 355 total NR.
2026-08-13 17:14:48,765 - INFO - Channels: 387 total FR | 387 total NR.
2026-08-13 17:14:49,055 - INFO - Channels: 532 total FR | 532 total NR.
2026-08-13 17:14:49,340 - INFO - Channels: 607 total FR | 607 total NR.
2026-08-13 17:14:49,627 - INFO - Cloud free segment: 1. Time: 2024-08-21T01:12:00.000000 - 2024-08-21T02:02:00.000000.
2026-08-13 17:14:49,628 - INFO - Channels: 355 total FR | 355 total NR.
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\misc\

In [40]:
## Aggregate overlap corrected profiles
data_cube.aggregate_profiles(var=['sigOLCor', 'BGOLCor'])

2026-08-13 17:20:11,607 - INFO - Aggregating variable: sigOLCor ...
2026-08-13 17:20:11,657 - INFO - Aggregating variable: BGOLCor ...


In [27]:
## Klett retrieval for overlap corrected profiles
data_cube.retrievalKlett(oc=True)

## Raman retrieval for overlap corrected profiles
data_cube.retrievalRaman(oc=True)

2026-08-13 17:14:56,378 - INFO - Klett retrieval for FR None overlap corrected signal ...
2026-08-13 17:14:56,379 - WARNING - rayleighfit seems to use range in matlab, but the met data should be in height >> RECHECK!
2026-08-13 17:14:56,380 - WARNING - at 10km height this is a difference of about 4 indices
2026-08-13 17:14:56,381 - INFO - Cloud free segment 0, Time: 2024-08-21T00:00:00.000000 - 2024-08-21T00:56:30.000000.
2026-08-13 17:14:56,381 - INFO - Channel: 532, total, FR klett.
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\klettfernald.py:276: RuntimeWarning: invalid value encountered in scalar divide
  denominator1 = RCS[iAlt + 1] / (aerBsc[iAlt + 1] + molBsc[iAlt + 1])
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\klettfernald.py:310: RuntimeWarning: divide by zero encountered in divide
  aerRelBRStd = np.abs((1 + noise / signal) / (1 + aerBsc + molBsc / 1e3) - 1)
2026-08-13 17:14:56,411 - INFO - Channel: 355, total, FR klett.
2026-08-13 17:14

## Depol and Ångström Profiles

Retrieval of Volume and Particle depolarization as well as Ångström 355/532 backscatter 532/1064 backscatter, and 355/532 Extinction

In [28]:
## Volume and particle depolarization
data_cube.calcDepol()

2026-08-13 17:15:00,294 - INFO - Calculate volume and particle depolarization ratios for product klett ...
2026-08-13 17:15:00,296 - INFO - voldepol at channel 532 cldFree 0 (np.int64(0), np.int64(114))
2026-08-13 17:15:00,298 - INFO - G [1.] [1.] H [-0.01477] [-0.9987] Eta 12.066747512729577 error [ 0.00027  0.02024 -0.00463] Window 25 
2026-08-13 17:15:00,300 - INFO - G [1.] [1.] H [-0.01477] [-0.9987] Eta 12.066747512729577 error [ 0.00027  0.02024 -0.00463] Window 1 
2026-08-13 17:15:00,302 - INFO - voldepol at channel 355 cldFree 0 (np.int64(0), np.int64(114))
2026-08-13 17:15:00,304 - INFO - G [1.] [1.] H [0.02041] [-0.998] Eta 47.27529323156868 error [ 0.00016  0.01567 -0.00958] Window 25 
2026-08-13 17:15:00,307 - INFO - G [1.] [1.] H [0.02041] [-0.998] Eta 47.27529323156868 error [ 0.00016  0.01567 -0.00958] Window 1 
2026-08-13 17:15:00,308 - INFO - voldepol at channel 1064 cldFree 0 (np.int64(0), np.int64(114))
2026-08-13 17:15:00,310 - INFO - G [1.] [1.] H [-0.02439] [-0.99

In [29]:
## Ångström ratios
data_cube.Angstroem()

2026-08-13 17:15:00,470 - INFO - Calculate Angstrom exponents for product klett ...
2026-08-13 17:15:00,471 - INFO - Channels: 355_total_FR, 532_total_FR. Product: Bsc.
2026-08-13 17:15:00,473 - INFO - Channels: 355_total_NR, 532_total_NR. Product: Bsc.
2026-08-13 17:15:00,475 - INFO - Channels: 532_total_FR, 1064_total_FR. Product: Bsc.
2026-08-13 17:15:00,476 - INFO - Channels: 355_total_FR, 532_total_FR. Product: Ext.
2026-08-13 17:15:00,477 - INFO - Channels: 355_total_NR, 532_total_NR. Product: Ext.
2026-08-13 17:15:00,478 - INFO - Channels: 355_total_FR, 532_total_FR. Product: Bsc.
2026-08-13 17:15:00,480 - INFO - Channels: 355_total_NR, 532_total_NR. Product: Bsc.
2026-08-13 17:15:00,481 - INFO - Channels: 532_total_FR, 1064_total_FR. Product: Bsc.
2026-08-13 17:15:00,482 - INFO - Channels: 355_total_FR, 532_total_FR. Product: Ext.
2026-08-13 17:15:00,483 - INFO - Channels: 355_total_NR, 532_total_NR. Product: Ext.
2026-08-13 17:15:00,484 - INFO - Calculate Angstrom exponents fo

## Lidar Calibration

- Lidar calibration constants (LC) are retieved for each channel at each cloud free period for both Klett and Raman retieved profiles
- All retrieved LCs are stored in the database
- If no LCs can be retrieved for a given channel, the LCs in the time range [24h before the measurment, 24h after the measurement] for the given channel included in the database will be used
- The optimal LC, ie. the one with the lowest standard deviation per channel is used for the processing.
- The following priority order is used when choosing the optimal LC
    1. Raman retrieved LC from data
    2. Klett retrieved LC from data
    3. Raman retrieved LC from database
    4. Klett retrieved LC from database

In [30]:
## Lidar calibration for both Klett and Raman retrieval
data_cube.LidarCalibration(db_path=DATABASE_PATH)

2026-08-13 17:15:00,518 - INFO - Calculating lidar calibration constants ...
2026-08-13 17:15:00,520 - INFO - LC retrieval: klett method
2026-08-13 17:15:00,530 - INFO - Using Retrieved Exticntion
2026-08-13 17:15:00,591 - INFO - cldFreGrp 0, Channel 532 total FR, LC_stable 110480476774235.88, LCStd 0.0022347503798313562
2026-08-13 17:15:00,598 - INFO - Using Retrieved Exticntion
2026-08-13 17:15:00,659 - INFO - cldFreGrp 0, Channel 355 total FR, LC_stable 25663860675124.36, LCStd 0.0016978384613928579
2026-08-13 17:15:00,667 - INFO - Using Retrieved Exticntion
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\misc\helper.py:1256: RuntimeWarning: Mean of empty slice
  thisMean = np.nanmean(window)
2026-08-13 17:15:00,731 - INFO - cldFreGrp 0, Channel 1064 total FR, LC_stable 70615096577805.12, LCStd 0.003027279079923954
2026-08-13 17:15:00,740 - INFO - Using Retrieved Exticntion
2026-08-13 17:15:00,798 - INFO - cldFreGrp 0, Channel 532 total NR, LC_stable 7336742488577.648, LCStd 0.0

In [31]:
## Display Lidar calibration constants per channel
data_cube.LCused

{'1064_total_FR': np.float64(50812275764219.61),
 '355_total_FR': np.float64(17022098372209.125),
 '355_total_NR': np.float64(2755558268816.5083),
 '532_total_FR': np.float64(113784215905298.44),
 '532_total_NR': np.float64(8843202624859.408),
 '387_total_FR': np.float64(48925494588027.48),
 '387_total_NR': np.float64(4006008627669.4106),
 '607_total_FR': np.float64(290870505264845.75),
 '607_total_NR': np.float64(13494237887722.975)}

In [32]:
## Store calibration constants in database
data_cube.write_2_sql_db(db_path=str(DATABASE_PATH), parameter='LC', method='raman')
data_cube.write_2_sql_db(db_path=str(DATABASE_PATH), parameter='LC', method='klett')
data_cube.write_2_sql_db(db_path=str(DATABASE_PATH), parameter='DC')

2026-08-13 17:15:02,481 - INFO - writing to sqlite-db: PicassoPyDatabase.db
2026-08-13 17:15:02,482 - INFO - writing LC to table: lidar_calibration_constant
2026-08-13 17:15:02,490 - INFO - 18 rows inserted into 'lidar_calibration_constant'.
2026-08-13 17:15:02,491 - INFO - writing to sqlite-db: PicassoPyDatabase.db
2026-08-13 17:15:02,492 - INFO - writing LC to table: lidar_calibration_constant
2026-08-13 17:15:02,500 - INFO - 10 rows inserted into 'lidar_calibration_constant'.
2026-08-13 17:15:02,501 - INFO - writing to sqlite-db: PicassoPyDatabase.db
2026-08-13 17:15:02,502 - INFO - writing DC to table: depol_calibration_constant
2026-08-13 17:15:02,509 - INFO - 5 rows inserted into 'depol_calibration_constant'.


dict_keys(['klett', 'raman', 'klett_db', 'raman_db'])
dict_keys(['klett', 'raman', 'klett_db', 'raman_db'])
dict_keys(['klett', 'raman', 'klett_db', 'raman_db'])


In [41]:
## Save retrieved optical profiles
write_profile2nc_file(data_cube=data_cube, prod_ls=["profiles", "NR_profiles", "OC_profiles"])

2026-08-13 17:29:50,605 - INFO - saving product: profiles


2026-08-13 17:29:50,762 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_0000_0056_profiles.nc
2026-08-13 17:29:51,112 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_0112_0202_profiles.nc
2026-08-13 17:29:51,354 - INFO - saving product: NR_profiles
2026-08-13 17:29:51,497 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_0000_0056_NR_profiles.nc
2026-08-13 17:29:52,345 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_0112_0202_NR_profiles.nc
2026-08-13 17:29:52,410 - INFO - saving product: OC_profiles
2026-08-13 17:29:52,556 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_0000_0056_OC_profiles.nc
2026-08-13 17:29:52,873 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_0112_0202_OC_profiles.nc


## High Resoulution Retrievals

The following high resolution (30s) time-height data are retrieved: 

- Attenuated backscatter
- Volume depolarization
- Molecular backscatter and extinction
- Quality mask
- QuasiV1 and QuasiV2 retrievals
- Target categorization V1 and V2

In [34]:
## Highres attenuated backscatter and volume depolarization
data_cube.attBsc_volDepol()

## Highres molecular signal
data_cube.molecularHighres()

2026-08-13 17:15:03,954 - INFO - 2D attenuated backscatter retrieval ...
2026-08-13 17:15:04,887 - WARNING - Exprimental, attenuated backscatter solution for 387_total_NR
2026-08-13 17:15:05,089 - INFO - Channel: 355 total FR | 355 cross FR
2026-08-13 17:15:05,277 - INFO - G [1.] [1.] H [0.02041] [-0.998] Eta 47.27529323156868 error [ 0.00016  0.01567 -0.00958] Window 1 
2026-08-13 17:15:05,735 - INFO - Channel: 532 total FR | 532 cross FR
2026-08-13 17:15:05,947 - INFO - G [1.] [1.] H [-0.01477] [-0.9987] Eta 12.066747512729577 error [ 0.00027  0.02024 -0.00463] Window 1 
2026-08-13 17:15:06,440 - INFO - Channel: 1064 total FR | 1064 cross FR
2026-08-13 17:15:06,712 - INFO - G [1.] [1.] H [-0.02439] [-0.996] Eta 0.14061415579754508 error [ 0.00133  0.02379 -0.01205] Window 1 
2026-08-13 17:15:07,737 - INFO - 2D volume depolarization ratio retrieval ...
2026-08-13 17:15:07,971 - INFO - G [1.] [1.] H [-0.01477] [-0.9987] Eta 12.066747512729577 error [ 0.00027  0.02024 -0.00463] Window 1

In [35]:
## Quality mask of signal
data_cube.estQualityMask()

2026-08-13 17:15:11,347 - INFO - Estimate quality masks ...
2026-08-13 17:15:11,348 - INFO - Calculating SNR from smoothed signal to improve data quality...
2026-08-13 17:15:11,771 - INFO - Vectorized group 0, Nr = 3, Nc = 10 ...
2026-08-13 17:15:28,136 - INFO - Using improved SNR in quality mask estimation.


In [36]:
## QuasiV1 retrievals and Target categorization
data_cube.quasiV1()

2026-08-13 17:15:28,710 - INFO - Calculating Quasi V1 particle backscatter coefficient
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV1.py:134: RuntimeWarning: divide by zero encountered in divide
  quasi_par_bsc = att_beta / (mol_att * quasi_par_att)**2 - molBsc
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV1.py:134: RuntimeWarning: overflow encountered in divide
  quasi_par_bsc = att_beta / (mol_att * quasi_par_att)**2 - molBsc
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV1.py:136: RuntimeWarning: overflow encountered in multiply
  quasi_par_ext = quasi_par_bsc * LRaer
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV1.py:133: RuntimeWarning: overflow encountered in multiply
  quasi_par_att = np.exp(-np.nancumsum(quasi_par_ext * diff_height, axis=1))
c:\Users\buholdt\AppData\Local\miniconda3\envs\PicassoPy\Lib\site-packages\numpy\_core\fromnumeric.py:57: RuntimeWarning: overflow encountered in ac

In [37]:
## QuasiV2 retrievals and Target categorization
data_cube.quasiV2()

2026-08-13 17:15:54,232 - INFO - Calculating Quasi V2 particle backscatter coefficient
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV2.py:170: RuntimeWarning: divide by zero encountered in divide
  quasi_par_bsc = (att_beta_el / att_beta_ra) * quasi_par_att - molBscEl
c:\Users\buholdt\AppData\Local\miniconda3\envs\PicassoPy\Lib\site-packages\numpy\_core\fromnumeric.py:57: RuntimeWarning: invalid value encountered in accumulate
  return bound(*args, **kwds)
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV2.py:169: RuntimeWarning: overflow encountered in exp
  quasi_par_att = np.exp((1 - (wv / wv_r)**AE) * OD_par + (OD_mol - OD_mol_r)) * molBscEl
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV2.py:170: RuntimeWarning: invalid value encountered in multiply
  quasi_par_bsc = (att_beta_el / att_beta_ra) * quasi_par_att - molBscEl
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV2.py:167: RuntimeWarning: ov

In [38]:
## Save highres retrievals
write2nc_file(data_cube=data_cube, prod_ls=["att_bsc", "NR_att_bsc", "OC_att_bsc", "vol_depol", "quasi_results", "quasi_results_V2", "target_classification", "target_classification_V2"])

2026-08-13 17:16:28,228 - INFO - saving product: att_bsc
2026-08-13 17:16:28,699 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_att_bsc.nc
2026-08-13 17:16:39,307 - INFO - saving product: NR_att_bsc
2026-08-13 17:16:39,684 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_NR_att_bsc.nc
2026-08-13 17:16:46,948 - INFO - saving product: OC_att_bsc
2026-08-13 17:16:47,056 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_OC_att_bsc.nc
2026-08-13 17:16:53,553 - INFO - saving product: vol_depol
2026-08-13 17:16:53,677 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_vol_depol.nc
2026-08-13 17:16:58,634 - INFO - saving product: quasi_results
2026-08-13 17:16:58,736 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_quasi_results.nc
2026-08-13 17:17:04,521 - INFO - saving product: quasi_results_V2
2026-08-13 17:17:04,659 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_qu